# nemo-endpoints-test

Systematic HTTP endpoint tester for all NeMo Microservices APIs on the Miramar platform.

**Requires SSH tunnel:** `ssh -L 8082:localhost:8082 -L 8888:localhost:8888 spark-79b7.local`  
**Requires laptop `/etc/hosts`:** `127.0.0.1 nemo.test nim.test data-store.test`

Hosts under test:
- `nemo.test:8082` — NeMo Microservices (entity store, deployment, customizer, evaluator, etc.)
- `nim.test:8082` — NIM inference gateway (OpenAI-compatible; only if a model is deployed)
- `data-store.test:8082` — HuggingFace-compatible data/model store

Run all cells top-to-bottom. The final cell prints a summary table.

In [ ]:
import requests
import json
from datetime import datetime

# Adjust if needed:
#   8082  — from JupyterLab on DGX (NeMo ingress port-forward)
#   80    — if running curl directly on DGX host
PORT = 8082

NEMO_BASE = f'http://nemo.test:{PORT}'
NIM_BASE  = f'http://nim.test:{PORT}'
DS_BASE   = f'http://data-store.test:{PORT}'

TIMEOUT = 10  # seconds per request

results = []  # list of (label, http_status_or_error, verdict)

def test(label, method, url, **kwargs):
    """Make a request, print a one-liner, store result. Returns (status_str, verdict)."""
    try:
        resp = getattr(requests, method.lower())(url, timeout=TIMEOUT, **kwargs)
        code = resp.status_code
        ok = 200 <= code < 300
        verdict = 'PASS' if ok else 'FAIL'
        try:
            snippet = json.dumps(resp.json())[:300]
        except Exception:
            snippet = resp.text[:300]
        print(f'[{verdict}] {method.upper()} {url}  →  HTTP {code}')
        print(f'        {snippet}')
        status_str = str(code)
    except requests.exceptions.ConnectionError as e:
        verdict, status_str = 'SKIP', 'CONN_ERR'
        print(f'[SKIP] {method.upper()} {url}  →  connection error (service not deployed?)')
    except requests.exceptions.Timeout:
        verdict, status_str = 'SKIP', 'TIMEOUT'
        print(f'[SKIP] {method.upper()} {url}  →  timed out after {TIMEOUT}s')
    except Exception as e:
        verdict, status_str = 'SKIP', f'ERR:{type(e).__name__}'
        print(f'[SKIP] {method.upper()} {url}  →  {e}')
    results.append((label, status_str, verdict))
    return status_str, verdict

print(f'Setup complete — {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
print(f'  NEMO:       {NEMO_BASE}')
print(f'  NIM:        {NIM_BASE}')
print(f'  Data Store: {DS_BASE}')

## nemo.test — Entity Store

Core registry: namespaces, projects, datasets, repos, and model registry.

In [ ]:
print('=== Entity Store ===')
test('GET /v1/namespaces', 'GET', f'{NEMO_BASE}/v1/namespaces')
test('GET /v1/projects',   'GET', f'{NEMO_BASE}/v1/projects')
test('GET /v1/datasets',   'GET', f'{NEMO_BASE}/v1/datasets')
test('GET /v1/repos',      'GET', f'{NEMO_BASE}/v1/repos')
test('GET /v1/models',     'GET', f'{NEMO_BASE}/v1/models')

## nemo.test — Deployment Service

NIM model deployment registry and inference gateway catalog.

In [ ]:
print('=== Deployment Service ===')
test('GET /v1/deployment/model-deployments', 'GET', f'{NEMO_BASE}/v1/deployment/model-deployments')
test('GET /v2/inference/gateway',            'GET', f'{NEMO_BASE}/v2/inference/gateway')
test('GET /v2/inference',                    'GET', f'{NEMO_BASE}/v2/inference')
test('GET /v2/models',                       'GET', f'{NEMO_BASE}/v2/models')

## nemo.test — Core API (Jobs)

In [ ]:
print('=== Core API ===')
test('GET /v1/jobs', 'GET', f'{NEMO_BASE}/v1/jobs')

## nemo.test — Customizer

Fine-tuning job submission and base model listing.

In [ ]:
print('=== Customizer ===')
test('GET /v1/customization/jobs',   'GET', f'{NEMO_BASE}/v1/customization/jobs')
test('GET /v1/customization/models', 'GET', f'{NEMO_BASE}/v1/customization/models')

## nemo.test — Evaluator

Evaluation jobs and benchmark targets. Both v1 and v2 route to `nemo-evaluator:7331`.

In [ ]:
print('=== Evaluator ===')
test('GET /v1/evaluation/jobs',    'GET', f'{NEMO_BASE}/v1/evaluation/jobs')
test('GET /v1/evaluation/targets', 'GET', f'{NEMO_BASE}/v1/evaluation/targets')
test('GET /v2/evaluation/jobs',    'GET', f'{NEMO_BASE}/v2/evaluation/jobs')

## nemo.test — Data Designer

Synthetic data generation jobs.

In [ ]:
print('=== Data Designer ===')
test('GET /v1/data-designer/jobs', 'GET', f'{NEMO_BASE}/v1/data-designer/jobs')

## nemo.test — Auditor, Safe Synthesizer & Intake

In [ ]:
print('=== Auditor / Safe Synthesizer / Intake ===')
test('GET /v1beta1/audit',             'GET', f'{NEMO_BASE}/v1beta1/audit')
test('GET /v1beta1/safe-synthesizer',  'GET', f'{NEMO_BASE}/v1beta1/safe-synthesizer')
test('GET /v1/intake',                 'GET', f'{NEMO_BASE}/v1/intake')

## data-store.test — HuggingFace-Compatible Data Store

Backed by MinIO inside minikube. Set `HF_ENDPOINT=http://data-store.test:8082/v1/hf` to route HF tooling here.

In [ ]:
print('=== Data Store ===')
test('GET /v1/health (data-store)', 'GET', f'{DS_BASE}/v1/health')

## nim.test — NIM Inference (OpenAI-Compatible)

Routes to `nemo-nim-proxy`. Only active when a NIM model is deployed.  
This section probes `/v1/models` first; if a model is found it dynamically runs a minimal chat completion.

In [ ]:
print('=== NIM Inference ===')

# Always test model list
nim_model = None
try:
    r = requests.get(f'{NIM_BASE}/v1/models', timeout=TIMEOUT)
    code = r.status_code
    ok = 200 <= code < 300
    verdict = 'PASS' if ok else 'FAIL'
    try:
        body = r.json()
        snippet = json.dumps(body)[:300]
        model_list = body.get('data', [])
        if model_list:
            nim_model = model_list[0].get('id')
    except Exception:
        snippet = r.text[:300]
    print(f'[{verdict}] GET {NIM_BASE}/v1/models  →  HTTP {code}')
    print(f'        {snippet}')
    results.append(('GET /v1/models (NIM)', str(code), verdict))
except requests.exceptions.ConnectionError:
    print(f'[SKIP] GET {NIM_BASE}/v1/models  →  connection error (NIM not deployed?)')
    results.append(('GET /v1/models (NIM)', 'CONN_ERR', 'SKIP'))
except requests.exceptions.Timeout:
    print(f'[SKIP] GET {NIM_BASE}/v1/models  →  timed out')
    results.append(('GET /v1/models (NIM)', 'TIMEOUT', 'SKIP'))

# Chat completions — dynamic model if available, otherwise skip
if nim_model:
    print(f'\nDeployed model: {nim_model}')
    test(
        'POST /v1/chat/completions (NIM)', 'POST',
        f'{NIM_BASE}/v1/chat/completions',
        json={
            'model': nim_model,
            'messages': [{'role': 'user', 'content': 'Say hello in one word.'}],
            'max_tokens': 10,
        },
        headers={'Content-Type': 'application/json'},
    )
else:
    print('No deployed NIM model found — skipping chat completions')
    results.append(('POST /v1/chat/completions (NIM)', 'SKIPPED', 'SKIP'))

# Completions and embeddings — attempt regardless (may 422 with no model)
test(
    'POST /v1/completions (NIM)', 'POST',
    f'{NIM_BASE}/v1/completions',
    json={'model': nim_model or '', 'prompt': 'Hello', 'max_tokens': 5},
    headers={'Content-Type': 'application/json'},
)
test(
    'POST /v1/embeddings (NIM)', 'POST',
    f'{NIM_BASE}/v1/embeddings',
    json={'model': nim_model or '', 'input': 'test embedding'},
    headers={'Content-Type': 'application/json'},
)

## Summary

In [ ]:
W = 50
print(f'\n{"="*70}')
print('ENDPOINT TEST SUMMARY')
print(f'{"="*70}')
print(f'{"Endpoint":<{W}} {"Status":<10} Result')
print(f'{"-"*W} {"-"*10} ------')

passed = failed = skipped = 0
for label, status, verdict in results:
    print(f'{label:<{W}} {status:<10} {verdict}')
    if verdict == 'PASS':   passed  += 1
    elif verdict == 'FAIL': failed  += 1
    else:                   skipped += 1

print(f'{"="*70}')
print(f'PASS: {passed}   FAIL: {failed}   SKIP: {skipped}   TOTAL: {len(results)}')
print(f'{"="*70}')

if failed:
    print(f'\n{failed} endpoint(s) returned unexpected HTTP errors.')
elif skipped == len(results):
    print('\nAll tests skipped — check SSH tunnel and /etc/hosts.')
else:
    print('\nAll reachable endpoints passed.')